In [ ]:
!pip install langchain langchain-community langchain-openai
!pip install chromadb sentence-transformers
!pip install pypdf pandas markdown
!pip install rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.19
    Uninstalling langchain-core-1.2.19:
      Successfully uninstalled langchain-core-1.2.19
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    #old model model="nvidia/nemotron-nano-9b-v2:free",
    #NEW MODEL
    model="nvidia/nemotron-3-super-120b-a12b:free",
    # old api key api_key="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",
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.2,
    #new api key
    api_key="API_1"
)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_loader = PyPDFLoader("Fraud Detection.pdf")
pdf_docs = pdf_loader.load()

In [ ]:
import pandas as pd

df = pd.read_csv("Fraud Detection - Sheet1.csv")

csv_docs = df.to_dict(orient="records")

In [ ]:
from langchain_community.document_loaders import TextLoader

md_loader = TextLoader("Fraud Detection.md")
md_docs = md_loader.load()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

pdf_chunks = splitter.split_documents(pdf_docs)
md_chunks = splitter.split_documents(md_docs)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_949/2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=pdf_chunks + md_chunks,
    embedding=embeddings,
    persist_directory="vectordb"
)

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":5}
)

In [ ]:
def factual_qa(query):

    docs = vectorstore.similarity_search(query, k=5)

    context = "\n".join([d.page_content for d in docs])

    prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}

Also cite the source filenames.
"""

    return llm.invoke(prompt)

In [ ]:
def summarize_docs(topic):

    docs = vectorstore.similarity_search(topic, k=5)

    text = "\n".join([d.page_content for d in docs])

    prompt = f"""
Summarize the following information:

{text}

Provide a clear structured summary.
"""

    return llm.invoke(prompt)

In [ ]:
def comparative_analysis(query):

    docs = vectorstore.similarity_search(query, k=5)

    context = "\n".join([d.page_content for d in docs])

    prompt = f"""
Analyze and compare information across the sources based on the following question.

Context:
{context}

Question:
{query}

Highlight relationships and differences. Specifically, look for and include any numerical performance metrics like accuracy, precision, and recall from the context. Present these numerical comparisons in a clear, structured way, ideally in a table format.
"""

    return llm.invoke(prompt)

In [ ]:
def classify_query(query):

    prompt = f"""
Classify the user query into one category:

1. factual
2. comparative
3. summary

Query: {query}

Return only the category.
"""

    return llm.invoke(prompt).content.strip()

In [ ]:
def agent(query):

    intent = classify_query(query)

    if "factual" in intent:
        return factual_qa(query)

    elif "comparative" in intent:
        return comparative_analysis(query)

    elif "summary" in intent:
        return summarize_docs(query)

    else:
        return factual_qa(query)

In [ ]:
query = input("Enter your question: ")

response = agent(query)

print("\nAnswer:\n")
print(response)

KeyboardInterrupt: Interrupted by user

In [ ]:
from langchain_core.documents import Document
import pandas as pd

# Load CSV
csv_df = pd.read_csv("Fraud Detection - Sheet1.csv")
csv_records = csv_df.to_dict(orient="records")

csv_documents = []
for row in csv_records:
    content = (
        f"Transaction ID: {row['transaction_id']}, "
        f"Amount: {row['amount']}, "
        f"Merchant: {row['merchant']}, "
        f"User ID: {row['user_id']}, "
        f"Is Fraud: {row['is_fraud']}"
    )
    metadata = {"source": "transactions.csv", "transaction_id": row['transaction_id']}
    csv_documents.append(Document(page_content=content, metadata=metadata))

In [ ]:
csv_chunks = splitter.split_documents(csv_documents)

vectorstore.add_documents(csv_chunks)

print("CSV data has been successfully added to the vector store.")

CSV data has been successfully added to the vector store.


In [ ]:
!pip install gradio
!pip install langchain langchain-community
!pip install chromadb sentence-transformers
!pip install pypdf

In [ ]:
%%writefile gradio_app.py

import gradio as gr
import os
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader, TextLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# Set the API key from environment variable
# old api os.environ["OPENROUTER_API_KEY"] = "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"
os.environ["OPENROUTER_API_KEY"] ="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"
# --- LLM Initialization (Cached) ---
def initialize_llm():
    return ChatOpenAI(
        #old model model="        nvidia/nemotron-3-super-120b-a12b:free",

        model="nvidia/nemotron-3-super-120b-a12b:free",
        openai_api_key=os.getenv("OPENROUTER_API_KEY"),
        openai_api_base="https://openrouter.ai/api/v1",
        temperature=0.2
    )

llm = initialize_llm()

# --- Agent Functions ---
def factual_qa(query, vectorstore):
    docs = vectorstore.similarity_search(query, k=5)
    context = "\n".join([d.page_content for d in docs])
    prompt_template = "Answer the question using the context below.\n\nContext:\n{context}\n\nQuestion:\n{query}\n\nAlso cite the source filenames."
    prompt = prompt_template.format(context=context, query=query)
    return llm.invoke(prompt).content

def summarize_docs(topic, vectorstore):
    docs = vectorstore.similarity_search(topic, k=5)
    text = "\n".join([d.page_content for d in docs])
    prompt_template = "Summarize the following information:\n\n{text}\n\nProvide a clear structured summary."
    prompt = prompt_template.format(text=text)
    return llm.invoke(prompt).content

def comparative_analysis(query, vectorstore):
    docs = vectorstore.similarity_search(query, k=5)
    context = "\n".join([d.page_content for d in docs])
    prompt_template = "Analyze and compare information across the sources based on the following question.\n\nContext:\n{context}\n\nQuestion:\n{query}\n\nHighlight relationships and differences. Specifically, look for and include any numerical performance metrics like accuracy, precision, and recall from the context. Present these numerical comparisons in a clear, structured way, ideally in a table format."
    prompt = prompt_template.format(context=context, query=query)
    return llm.invoke(prompt).content

def classify_query(query):
    prompt_template = "Classify the user query into one category:\n\n1. factual\n2. comparative\n3. summary\n\nQuery: {query}\n\nReturn only the category."
    prompt = prompt_template.format(query=query)
    return llm.invoke(prompt).content.strip()

def agent(query, vectorstore):
    intent = classify_query(query)
    if "factual" in intent:
        return factual_qa(query, vectorstore)
    elif "comparative" in intent:
        return comparative_analysis(query, vectorstore)
    elif "summary" in intent:
        return summarize_docs(query, vectorstore)
    else:
        return factual_qa(query, vectorstore)

# --- Gradio Interface Function ---
def process_uploaded_files(files, user_query):
    if not files:
        return "Please upload at least one document."

    all_docs = []
    for file in files:
        try:
            filepath = file.name
            if filepath.endswith(".pdf"):
                loader = PyPDFLoader(filepath)
            elif filepath.endswith(".md") or filepath.endswith(".txt"):
                loader = TextLoader(filepath)
            elif filepath.endswith(".csv"):
                # For CSV, load into pandas first to convert to a consistent Document format
                df = pd.read_csv(filepath)
                csv_records = df.to_dict(orient="records")
                csv_documents = []
                for row in csv_records:
                    content = ", ".join([f"{k}: {v}" for k, v in row.items()])
                    metadata = {"source": os.path.basename(filepath)}
                    csv_documents.append(Document(page_content=content, metadata=metadata))
                all_docs.extend(csv_documents)
                continue # Skip generic loader for CSV
            else:
                return f"Unsupported file type: {os.path.basename(filepath)}"
            all_docs.extend(loader.load())
        except Exception as e:
            return f"Error processing {os.path.basename(filepath)}: {e}"

    # Chunk documents
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100
    )
    chunks = splitter.split_documents(all_docs)

    # Create embeddings and vector store
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    vectorstore = Chroma.from_documents(chunks, embeddings)

    # Invoke the agent with the dynamically created vectorstore
    try:
        response = agent(user_query, vectorstore)
        return response
    except Exception as e:
        return f"An error occurred during response generation: {e}"

# --- Gradio Interface ---
iface = gr.Interface(
    fn=process_uploaded_files,
    inputs=[
        gr.File(file_count="multiple", label="Upload PDF, CSV, or Markdown files"),
        gr.Textbox(lines=3, placeholder="Enter your question here...", label="Question")
    ],
    outputs=gr.Textbox(lines=25, label="Answer", show_copy_button=True),
    title="AI Knowledge Assistant (Gradio)",
    description="Upload documents and ask questions based on their content."
)
if __name__ == "__main__":
    iface.launch(share=True)


Writing gradio_app.py


In [ ]:
!python gradio_app.py

Traceback (most recent call last):
  File "/content/gradio_app.py", line 2, in <module>
    import gradio as gr
  File "/usr/local/lib/python3.12/dist-packages/gradio/__init__.py", line 11, in <module>
    from gradio.cli import deploy
  File "/usr/local/lib/python3.12/dist-packages/gradio/cli/__init__.py", line 1, in <module>
    from .cli import cli, deploy
  File "/usr/local/lib/python3.12/dist-packages/gradio/cli/cli.py", line 9, in <module>
    from .commands import (
  File "/usr/local/lib/python3.12/dist-packages/gradio/cli/commands/__init__.py", line 5, in <module>
    from .load import load_app
  File "/usr/local/lib/python3.12/dist-packages/gradio/cli/commands/load.py", line 5, in <module>
    from .load_chat import main as chat
object address  : 0x7cb950016c80
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C


ADDING CONFIDENCE SCORE AND CITATIONS

In [ ]:
%%writefile gradio_app.py

import gradio as gr
import os
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document


os.environ["OPENROUTER_API_KEY"] = "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"

# --- LLM Initialization ---
def initialize_llm():
    return ChatOpenAI(
        model="nvidia/nemotron-3-super-120b-a12b:free",
        openai_api_key=os.getenv("OPENROUTER_API_KEY"),
        openai_api_base="https://openrouter.ai/api/v1",
        temperature=0.2
    )

llm = initialize_llm()

# --- Confidence Function ---
def get_confidence(docs):
    if len(docs) >= 4:
        return "High"
    elif len(docs) >= 2:
        return "Medium"
    else:
        return "Low"

# --- Helper: Build Context with Sources ---
def build_context(docs):
    return "\n\n".join([
        f"Source: {d.metadata.get('source', 'unknown')} | Chunk: {i}\nContent: {d.page_content}"
        for i, d in enumerate(docs)
    ])

# --- Tools ---
def factual_qa(query, vectorstore):
    docs = vectorstore.similarity_search(query, k=5)
    confidence = get_confidence(docs)
    context = build_context(docs)

    prompt = f"""
    Answer the question using ONLY the context below.

    Context:
    {context}

    Question:
    {query}

    At the end, list all source filenames used.
    """

    answer = llm.invoke(prompt).content
    return answer + f"\n\nConfidence: {confidence}"

def summarize_docs(query, vectorstore):
    docs = vectorstore.similarity_search(query, k=5)
    confidence = get_confidence(docs)
    context = build_context(docs)

    prompt = f"""
    Summarize the following content:

    {context}

    Provide a structured summary.
    Also list source filenames.
    """

    answer = llm.invoke(prompt).content
    return answer + f"\n\nConfidence: {confidence}"

def comparative_analysis(query, vectorstore):
    docs = vectorstore.similarity_search(query, k=5)
    confidence = get_confidence(docs)
    context = build_context(docs)

    prompt = f"""
    Analyze and compare the information below.

    Context:
    {context}

    Question:
    {query}

    Include numerical comparisons (accuracy, precision, recall).
    Present results clearly (table if possible).
    Also list source filenames.
    """

    answer = llm.invoke(prompt).content
    return answer + f"\n\nConfidence: {confidence}"

# --- Intent Classification ---
def classify_query(query):
    prompt = f"""
    Classify the query into one of:
    - factual
    - comparative
    - summary

    Only return ONE word.

    Query: {query}
    """
    intent = llm.invoke(prompt).content.lower().strip()

    if "comparative" in intent:
        return "comparative"
    elif "summary" in intent:
        return "summary"
    else:
        return "factual"

# --- Agent Dispatcher ---
def agent(query, vectorstore):
    intent = classify_query(query)

    if intent == "factual":
        return "Tool Used: Factual Q&A\n\n" + factual_qa(query, vectorstore)

    elif intent == "comparative":
        return "Tool Used: Comparative Analysis\n\n" + comparative_analysis(query, vectorstore)

    elif intent == "summary":
        return "Tool Used: Summarization\n\n" + summarize_docs(query, vectorstore)

    else:
        return "Tool Used: Factual Q&A\n\n" + factual_qa(query, vectorstore)

# --- Main Processing Function ---
def process_uploaded_files(files, user_query):
    if not files:
        return "Please upload at least one document."

    all_docs = []

    for file in files:
        try:
            filepath = file.name

            if filepath.endswith(".pdf"):
                loader = PyPDFLoader(filepath)
                docs = loader.load()

            elif filepath.endswith(".md") or filepath.endswith(".txt"):
                loader = TextLoader(filepath)
                docs = loader.load()

            elif filepath.endswith(".csv"):
                df = pd.read_csv(filepath)
                docs = []
                for i, row in df.iterrows():
                    content = ", ".join([f"{k}: {v}" for k, v in row.items()])
                    docs.append(Document(
                        page_content=content,
                        metadata={"source": os.path.basename(filepath)}
                    ))
            else:
                return f"Unsupported file type: {os.path.basename(filepath)}"

            # Add source metadata if missing
            for d in docs:
                if "source" not in d.metadata:
                    d.metadata["source"] = os.path.basename(filepath)

            all_docs.extend(docs)

        except Exception as e:
            return f"Error processing {os.path.basename(filepath)}: {e}"

    # --- Chunking ---
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100
    )
    chunks = splitter.split_documents(all_docs)

    # --- Embeddings + Vector Store ---
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vectorstore = Chroma.from_documents(chunks, embeddings)

    # --- Run Agent ---
    try:
        response = agent(user_query, vectorstore)
        return response
    except Exception as e:
        return f"Error generating response: {e}"

# --- Gradio UI ---
iface = gr.Interface(
    fn=process_uploaded_files,
    inputs=[
        gr.File(file_count="multiple", label="Upload PDF, CSV, or Markdown files"),
        gr.Textbox(lines=4, label="Enter your question")
    ],
    outputs=gr.Textbox(lines=25, label="AI Answer", show_copy_button=True),
    title="AI Knowledge Assistant",
    description="Upload documents and ask questions with source-based answers."
)

if __name__ == "__main__":
    iface.launch(share=True)

Overwriting gradio_app.py


In [ ]:
!python gradio_app.py

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://41f72bc0d843a8fee4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
/content/gradio_app.py:188: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100% 103/103 [00:00<00:00, 1088.89it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embe